# Webcam facial segmentation

**Bita Moharrami**

Run locally with a webcam and graphical desktop. This demonstration is not a benchmark of real-time throughput. Train a checkpoint using notebook 01 or supply the original HDF5 checkpoint as described in the README. Click the video window and press **q** to stop.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
from tensorflow.keras.models import load_model

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
MODEL_PATH = ROOT / "models" / "unet.keras"
# To use the original checkpoint instead, set MODEL_PATH to models/model.h5.
CAMERA_INDEX = 0
IMG_HEIGHT, IMG_WIDTH = 256, 256
COLORS_RGB = np.array([
    (0, 128, 128), (128, 0, 0), (255, 224, 189),
    (0, 255, 0), (0, 128, 0), (0, 0, 255),
    (0, 0, 128), (255, 0, 0), (255, 255, 255),
], dtype=np.uint8)
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Train a model first or place a checkpoint at {MODEL_PATH}")
model = load_model(MODEL_PATH, compile=False)

In [ ]:
cap = cv2.VideoCapture(CAMERA_INDEX)
try:
    if not cap.isOpened():
        raise RuntimeError("Cannot open webcam. Check camera permissions and CAMERA_INDEX.")
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        # Training decodes JPEG as RGB; OpenCV captures BGR.
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        image = cv2.resize(frame_rgb, (IMG_WIDTH, IMG_HEIGHT)).astype(np.float32) / 255.0
        probabilities = model.predict(image[None, ...], verbose=0)[0]
        labels = np.argmax(probabilities, axis=-1)
        mask_rgb = cv2.resize(COLORS_RGB[labels], (frame_bgr.shape[1], frame_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask_bgr = cv2.cvtColor(mask_rgb, cv2.COLOR_RGB2BGR)
        overlay_bgr = cv2.addWeighted(frame_bgr, 0.6, mask_bgr, 0.4, 0)
        cv2.imshow("Facial segmentation - press q to quit", overlay_bgr)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()